In [25]:
import openeo
import rasterio
from localtileserver import TileClient, get_leaflet_tile_layer
import leafmap

In [26]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")
connection.authenticate_oidc()

Authenticated using refresh token.


<Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>

#  Importing Shapefile from Syke
https://www.syke.fi/en/environmental-data/downloadable-spatial-datasets#peat-production-areas-after-use-category

In [27]:
import geopandas as gpd

In [28]:
peat = gpd.read_file("../data/raw/vectors/turvetuotantoalueet/turvetuotantoalueet_jalkikaytto.shp")

print(peat.crs)
print(len(peat))
print(peat.columns.tolist())
peat.head()

PROJCS["EUREF_FIN_TM35FIN",GEOGCS["EUREF-FIN",DATUM["European_Terrestrial_Reference_System_1989",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6258"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",27],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
164821
['OBJECTID', 'luokka', 'luokka_nro', 'Shape_Leng', 'Shape_Le_1', 'Shape_Area', 'geometry']


,OBJECTID,luokka,luokka_nro,Shape_Leng,Shape_Le_1,Shape_Area,geometry
0,1,avoin,5,19.387451,19.387451,22.538417,"POLYGON ((297344 6657056, 297344 6657051.677, ..."
1,2,avoin,5,57.283742,57.283742,55.436297,"POLYGON ((297336 6657080, 297336 6657063.652, ..."
2,3,avoin,5,29.051756,29.051756,20.186929,"POLYGON ((297376 6657088, 297363.251 6657088, ..."
3,4,avoin,5,24.240326,24.240326,24.311072,"POLYGON ((297392 6657096, 297383.813 6657096, ..."
4,5,avoin,5,15.272810,15.272810,9.650851,"POLYGON ((297400 6657104, 297394.842 6657104, ..."


In [29]:
# luokka = classification
# dropna = drop missing values
print(peat["luokka"].value_counts(dropna=False))

# avoin = open peat production areas
# metsittynyt = afforested peat production areas
# turvetuotanto = peat production areas
# vesi = water areas
# maatalous = agricultural areas
# rakennut = built-up areas

luokka
avoin            118758
metsittynyt       35181
turvetuotanto      5325
vesi               3078
maatalous          2478
rakennettu            1
Name: count, dtype: int64


In [30]:
#reproject to WGS84 for visualization
peat_wgs = peat.to_crs("EPSG:4326")

In [31]:
# visualising all peat areas from reprojected WGS84
# former peat production areas and the classification of what they are today
m = leafmap.Map(center=[63.0, 25.0], zoom=6)
m.add_gdf(peat_wgs, layer_name="Peat production areas")
m

Map(center=[63.0, 25.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out…

In [32]:
#reproject to WGS84 for visualization
peat_wgs = peat.to_crs("EPSG:4326")
# South Ostrobothnia and the area reaching south towards Satakunta and Pirkanmaa
study_area = {"west": 21.8, "south": 61.3, "east": 24.3, "north": 63.3}

peat_aoi = peat_wgs.cx[study_area["west"]:study_area["east"], study_area["south"]:study_area["north"]]
print(f"{len(peat_aoi)} peat production areas in the study area")


36359 peat production areas in the study area


In [33]:
# visualising all peat areas from study area (AOI)
m = leafmap.Map(center=[62.3, 23.0], zoom=8)
m.add_gdf(peat_aoi, layer_name="Peat production areas")
m

Map(center=[62.3, 23.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out…

In [34]:
# leafmap of the AOI with the after-use classes colour coded
# one colour per after-use class
colors = {
    "avoin": "#ff6600",        
    "metsittynyt": "#00e64d",  
    "turvetuotanto": "#cc00ff",
    "vesi": "#00ccff",         
    "maatalous": "#ffee00",    
    "rakennettu": "#ff0033",   
}

legend = {
    "Open": "#ff6600",
    "Afforested": "#00e64d",
    "Still in production": "#cc00ff",
    "Water": "#00ccff",
    "Agriculture": "#ffee00",
    "Built-up": "#ff0033",
}

# leafmap asks for a style per feature, so look the colour up by luokka
def style_by_class(feature):
    c = colors.get(feature["properties"]["luokka"], "#cccccc")
    return {"fillColor": c, "color": c, "fillOpacity": 0.8, "weight": 0}

m = leafmap.Map(center=[63.0, 25.0], zoom=6)
m.add_gdf(peat_aoi, layer_name="Peat production areas", style_callback=style_by_class)
m.add_legend(title="After-use class", legend_dict=legend)
m

Map(center=[63.0, 25.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out…

# Visualising Sentinel 2

In [35]:
from localtileserver import TileClient, get_leaflet_tile_layer
import leafmap

In [36]:
client = TileClient("../data/raw/raster/finland_test_rgb_2024.tif", host="127.0.0.1")
layer = get_leaflet_tile_layer(client, band=[1, 2, 3], vmin=0, vmax=3000)

# displaying the rgb raster map
m = leafmap.Map(center=[62.45, 22.6], zoom=11)
m.add_layer(layer)
m
m

Map(center=[62.45, 22.6], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

In [37]:
client = TileClient("../data/raw/raster/finland_test_rgb_2024.tif", host="127.0.0.1")
layer = get_leaflet_tile_layer(client, band=[1, 2, 3], vmin=0, vmax=3000)

# displaying the rgb raster map
m = leafmap.Map(center=[62.45, 22.6], zoom=11)
m.add_layer(layer)
m.add_gdf(peat_aoi, layer_name="Peat sites", style_callback=style_by_class)
m.add_legend(title="After-use class", legend_dict=legend)
m

Map(center=[62.45, 22.6], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

In [38]:
# Displaying the NDVI raster map
m = leafmap.Map(center=[62.45, 22.6], zoom=11)
m.add_raster("../data/raw/raster/finland_test_ndvi_2024.tif", colormap="RdYlGn", vmin=0, vmax=0.9, layer_name="NDVI")
m.add_gdf(peat_aoi, layer_name="Peat sites", style_callback=style_by_class)
m.add_legend(title="After-use class", legend_dict=legend)
m

Map(center=[62.449987, 22.602083999999998], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_i…

In [39]:
peat_test = peat_wgs.cx[22.4:22.8, 62.3:62.6]

m.add_gdf(peat_test, layer_name="Peat sites", style={"fillOpacity": 0, "color": "red", "weight": 1})
m

Map(center=[62.449987, 22.602083999999998], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_i…

In [40]:
# split map to compare RGB and NDVI
m = leafmap.Map(center=[62.45, 22.6], zoom=11, height="500px")
m.split_map(
    left_layer= "../data/raw/raster/finland_test_rgb_2024.tif",
    right_layer= "../data/raw/raster/finland_test_ndvi_2024.tif",
    left_args={"indexes": [1, 2, 3]},
    right_args={"vmin": 0, "vmax": 0.9, "colormap": "RdYlGn"},
    left_label="Sentinel-2 RGB (Summer 2024)",
    right_label="NDVI (Summer 2024)",
)

m.add_gdf(
    peat_aoi,
    layer_name="SYKE peat",
    style={"color": "magenta", "weight": 2, "fillOpacity": 0.2},
)

m

Map(center=[62.45, 22.6], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

# Othophotos - NLS

In [41]:
## MY API KEY NLS orthophoto
from pathlib import Path

# writes the key to a file that git ignores
# paste the key here, run once, then clear the line again
key_path = Path("../api_key.txt")
key_path.write_text("139d8588-3775-4d02-b0d2-68ff823e5d51")

print(f"saved to {key_path.resolve()}")

saved to C:\Users\clair\OneDrive\Dokumente\University Zurich\Semester 7 (HS)\sds320\project\api_key.txt


In [42]:
with open("../.gitignore", "a") as f:
    f.write("\napi_key.txt\n")

In [43]:
## Wrong base map
# read the key, which stays out of the notebook and out of git
nls_key = Path("../api_key.txt").read_text().strip()

# peat polygons for the test area
peat_aoi = peat_wgs.cx[22.4:22.8, 62.3:62.6]

m = leafmap.Map(center=[62.45, 22.6], zoom=13)

# NLS orthophoto as a background tile layer, nothing downloaded
m.add_wms_layer(
    url=f"https://avoin-karttakuva.maanmittauslaitos.fi/avoin/wmts?api-key={nls_key}",
    layers="ortokuva",
    name="NLS orthophoto",
    format="image/png",
)

# outline only, so the imagery stays visible underneath
def style_by_class(feature):
    c = colors.get(feature["properties"]["luokka"], "#cccccc")
    return {"fillColor": c, "color": c, "fillOpacity": 0, "weight": 2}

m.add_gdf(peat_aoi, layer_name="Peat sites", style_callback=style_by_class)
m.add_legend(title="After-use class", legend_dict=legend)
m

Map(center=[62.45, 22.6], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

In [44]:
# NLS orthophoto via WMTS tiles, note the z/y/x order
ortho_url = (
    "https://avoin-karttakuva.maanmittauslaitos.fi/avoin/wmts/1.0.0/"
    "ortokuva/default/WGS84_Pseudo-Mercator/{z}/{y}/{x}.jpg"
    f"?api-key={nls_key}"
)

m = leafmap.Map(center=[62.45, 22.6], zoom=14)
m.add_tile_layer(url=ortho_url, name="NLS orthophoto", attribution="National Land Survey of Finland")

m.add_gdf(peat_aoi, layer_name="Peat sites", style_callback=style_by_class)
m.add_legend(title="After-use class", legend_dict=legend)
m

Map(center=[62.45, 22.6], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

In [45]:
# NLS orthophoto via WMTS tiles, note the z/y/x order
ortho_url = (
    "https://avoin-karttakuva.maanmittauslaitos.fi/avoin/wmts/1.0.0/"
    "ortokuva/default/WGS84_Pseudo-Mercator/{z}/{y}/{x}.jpg"
    f"?api-key={nls_key}"
)

m = leafmap.Map(center=[62.45, 22.6], zoom=14)
m.add_tile_layer(url=ortho_url, name="NLS orthophoto", attribution="National Land Survey of Finland")

m.add_gdf(peat_aoi, layer_name="Peat sites", style_callback=style_by_class)
m.add_legend(title="After-use class", legend_dict=legend)
m

Map(center=[62.45, 22.6], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

In [46]:
# NLS orthophoto via WMTS tiles, note the z/y/x order
ortho_url = (
    "https://avoin-karttakuva.maanmittauslaitos.fi/avoin/wmts/1.0.0/"
    "ortokuva/default/WGS84_Pseudo-Mercator/{z}/{y}/{x}.jpg"
    f"?api-key={nls_key}"
)

m = leafmap.Map(center=[62.45, 22.6], zoom=14)
m.add_tile_layer(url=ortho_url, name="NLS orthophoto", attribution="National Land Survey of Finland")

m.add_gdf(peat_aoi, layer_name="Peat sites", style_callback=style_by_class)
m.add_legend(title="After-use class", legend_dict=legend)
m

Map(center=[62.45, 22.6], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…